In [42]:
import torch
class ContrastiveLoss(torch.nn.Module):

    def __init__(self, margin=1.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, input1, input2, y):
        diff = input1 - input2
        dist_sq = torch.sum(torch.pow(diff, 2), 1)
        dist = torch.sqrt(dist_sq)
        mdist = self.margin - dist
        dist = torch.clamp(mdist, min=0.0)
        loss = y * dist_sq + (1 - y) * torch.pow(dist, 2)
        loss = torch.sum(loss) / 2.0 / input1.size()[0]
        return loss

CAMBIAR: `__getitem__` debe devolver: (img1_file, img2_file, label)

In [58]:
from torch.utils.data import Dataset
from PIL import Image
import os

class LFWDataset(Dataset):
    def __init__(self, root_dir, path_file_dir, transform=None, random_aug=False):
        self.root_dir = root_dir
        self.transform = transform
        self.random_aug = random_aug
        self.random_aug_prob = 0.7

        data = []
        with open(path_file_dir, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                img1, img2, label = line.split(" ")
                data.append((img1, img2, int(label)))
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img1_rel, img2_rel, label = self.data[idx]
        img1 = Image.open(os.path.join(self.root_dir, img1_rel)).convert("RGB")
        img2 = Image.open(os.path.join(self.root_dir, img2_rel)).convert("RGB")

        # Si NO tienes random_augmentation implementado, deja random_aug=False
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, label


In [44]:
import torch.nn as nn

class Flatten(nn.Module):

    def forward(self, input):
        return input.view(input.size(0), -1)


In [45]:

class SiameseNetwork(nn.Module):

    def __init__(self, contra_loss=False):
        super(SiameseNetwork, self).__init__()

        self.contra_loss = contra_loss

        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5, padding=2, stride=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=5, padding=2, stride=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1, stride=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(256, 512, kernel_size=3, padding=1, stride=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(512),

            Flatten(),
            nn.Linear(131072, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(1024)
        )

        self.fc = nn.Sequential(
            nn.Linear(2048, 1),
            nn.Sigmoid()
        )

    def forward_once(self, x):
        output = self.cnn(x)
        return output

    def forward(self, input1, input2):
        output1 = self.forward_once(input1)
        output2 = self.forward_once(input2)
        if self.contra_loss:
            return output1, output2
        else:
            output = torch.cat((output1, output2), 1)
            output = self.fc(output)
            return output

In [46]:
import os
import random
import argparse
import time
from datetime import datetime
from pytz import timezone
import torch
import torch.nn as nn
import torchvision.datasets as dsets
import torchvision.transforms as transforms
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Variable
from PIL import Image
import cv2
BATCH_SIZE = 50

def cur_time():
    fmt = '%Y-%m-%d %H:%M:%S %Z%z'
    eastern = timezone('US/Eastern')
    naive_dt = datetime.now()
    loc_dt = datetime.now(eastern)
    return loc_dt.strftime(fmt).replace(' ', '_')

def threashold_sigmoid(t):
    """prob > 0.5 --> 1 else 0"""
    threashold = t.clone()
    threashold.data.fill_(0.5)
    return (t > threashold).float()


def threashold_contrastive_loss(input1, input2, m):
    """dist < m --> 1 else 0"""
    diff = input1 - input2
    dist_sq = torch.sum(torch.pow(diff, 2), 1)
    dist = torch.sqrt(dist_sq)
    threashold = dist.clone()
    threashold.data.fill_(m)
    return (dist < threashold).float().view(-1, 1)


def train(args):
    default_transform = transforms.Compose([
        transforms.Resize(128),
        transforms.ToTensor(),
    ])
    train_dataset = LFWDataset('output', 'train.txt', default_transform, args.randaug)
    print("Loaded {} training data.".format(len(train_dataset)))

    # # Data Loader (Input Pipeline)
    train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                               batch_size=BATCH_SIZE,
                                               shuffle=True)

    siamese_net = SiameseNetwork(args.contra_loss)
    if args.cuda:
        siamese_net = siamese_net.cuda()

    # # Loss and Optimizer
    if args.contra_loss:
        criterion = ContrastiveLoss(margin=args.margin)
    else:
        criterion = nn.BCELoss()

    optimizer = torch.optim.Adam(siamese_net.parameters())

    # Train the Model
    num_epochs = args.epoch
    for epoch in range(num_epochs):
        for i, (img1_set, img2_set, labels) in enumerate(train_loader):

            if args.cuda:
                img1_set = img1_set.cuda()
                img2_set = img2_set.cuda()
                labels = labels.cuda()

            img1_set = Variable(img1_set)
            img2_set = Variable(img2_set)
            labels = Variable(labels.view(-1, 1).float())

            # Forward + Backward + Optimize
            optimizer.zero_grad()
            if args.contra_loss:
                output1, output2 = siamese_net(img1_set, img2_set)
                loss = criterion(output1, output2, labels)
                loss.backward()
                optimizer.step()
            else:
                output_labels_prob = siamese_net(img1_set, img2_set)
                loss = criterion(output_labels_prob, labels)
                loss.backward()
                optimizer.step()
        print('Epoch [%d/%d], Iter [%d/%d] Loss: %.4f' % (epoch+1, num_epochs, i+1, len(train_dataset)//BATCH_SIZE, loss.item()))

    # Training accuracy
    test_against_data(args, 'training', train_loader, siamese_net)

    # Save the Trained Model
    model_file_name = "{}_{}".format(cur_time(), args.model_file)
    torch.save(siamese_net.state_dict(), model_file_name)
    print("Saved model at {}".format(model_file_name))
    return siamese_net


def test_against_data(args, label, dataset, siamese_net):
    # Training accuracy
    siamese_net.eval()  # Change model to 'eval' mode (BN uses moving mean/var).
    correct = 0.0
    total = 0.0
    for img1_set, img2_set, labels in dataset:
        labels = labels.view(-1, 1).float()
        if args.cuda:
            img1_set = img1_set.cuda()
            img2_set = img2_set.cuda()
            labels = labels.cuda()
        img1_set = Variable(img1_set)
        img2_set = Variable(img2_set)
        labels = Variable(labels)

        if args.contra_loss:
            output1, output2 = siamese_net(img1_set, img2_set)
            output_labels = threashold_contrastive_loss(output1, output2, args.margin)
        else:
            output_labels_prob = siamese_net(img1_set, img2_set)
            output_labels = threashold_sigmoid(output_labels_prob)

        if args.cuda:
            output_labels = output_labels.cuda()
        total += labels.size(0)
        correct += (output_labels == labels).sum().item()

    print('Accuracy of the model on the {} {} images: {} %%'.format(total, label, (100 * correct / total)))


def test(args, siamese_net=None):
    if not siamese_net:
        saved_model = torch.load(args.model_file)
        siamese_net = SiameseNetwork(args.contra_loss)
        siamese_net.load_state_dict(saved_model)

    if args.cuda:
        siamese_net = siamese_net.cuda()

    default_transform = transforms.Compose([
        transforms.Resize(128),
        transforms.ToTensor(),
    ])
    test_dataset = LFWDataset('output', 'test.txt', default_transform)
    print("Loaded {} test data.".format(len(test_dataset)))

    test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                              batch_size=BATCH_SIZE,
                                              shuffle=False)

    test_against_data(args, "testing", test_loader, siamese_net)


In [ ]:
import os
import itertools
import random
def generar_pairs_txt_lfw_from_users(root_dir, users, output_txt, subfolder="sobel", ext=".png"):
    imgs_por_usuario = {}

    for user in users:
        folder_abs = os.path.join(root_dir, user, subfolder)
        if not os.path.isdir(folder_abs):
            continue

        imgs_rel = sorted(
            os.path.join(user, subfolder, f).replace("\\", "/")
            for f in os.listdir(folder_abs)
            if f.lower().endswith(ext)
        )

        # Debug opcional
        # print(user, "->", len(imgs_rel), "imgs")
        # for p in imgs_rel[:5]:
        #     print(" ", p)

        if len(imgs_rel) >= 2:  # importante para tener positivos
            imgs_por_usuario[user] = imgs_rel

    users = sorted(imgs_por_usuario.keys())

    with open(output_txt, "w", encoding="utf-8") as f:
        # Positivos
        for user in users:
            for img1, img2 in itertools.combinations(imgs_por_usuario[user], 2):
                f.write(f"{img1} {img2} 1\n")

        # Negativos
        for i, u1 in enumerate(users):
            for u2 in users[i + 1:]:
                for img1 in imgs_por_usuario[u1]:
                    for img2 in imgs_por_usuario[u2]:
                        f.write(f"{img1} {img2} 0\n")

    print(f"{output_txt} generado con {len(users)} usuarios y {sum(len(v) for v in imgs_por_usuario.values())} imágenes")




train.txt generado con 8 usuarios y 18 imágenes
test.txt generado con 2 usuarios y 4 imágenes
root_dir: output
path_file_dir: train.txt
path_file: <_io.TextIOWrapper name='train.txt' mode='r' encoding='cp1252'>
root_dir: output
path_file_dir: test.txt
path_file: <_io.TextIOWrapper name='test.txt' mode='r' encoding='cp1252'>


In [59]:
import torchvision.transforms as transforms

default_transform = transforms.Compose([
    transforms.Resize(128),
    transforms.ToTensor(),
])

train_ds = LFWDataset(root_dir="output", path_file_dir="train.txt",
                      transform=default_transform, random_aug=False)

test_ds  = LFWDataset(root_dir="output", path_file_dir="test.txt",
                      transform=default_transform, random_aug=False)

print("train pairs:", len(train_ds))
print("test pairs:", len(test_ds))


train pairs: 153
test pairs: 6


In [62]:
from torch.utils.data import DataLoader

tmp_loader = DataLoader(train_ds, batch_size=4, shuffle=True)

img1_set, img2_set, labels = next(iter(tmp_loader))
print("img1_set:", type(img1_set), getattr(img1_set, "shape", None))
print("img2_set:", type(img2_set), getattr(img2_set, "shape", None))
print("labels:", type(labels), getattr(labels, "shape", None))

# mira también un sample directo
x1, x2, y = train_ds[0]
print("sample img1:", type(x1), getattr(x1, "shape", None))
print("sample img2:", type(x2), getattr(x2, "shape", None))
print("sample label:", y)


img1_set: <class 'torch.Tensor'> torch.Size([4, 3, 128, 128])
img2_set: <class 'torch.Tensor'> torch.Size([4, 3, 128, 128])
labels: <class 'torch.Tensor'> torch.Size([4])
sample img1: <class 'torch.Tensor'> torch.Size([3, 128, 128])
sample img2: <class 'torch.Tensor'> torch.Size([3, 128, 128])
sample label: 1


In [60]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def train_in_notebook(
    train_dataset,
    test_dataset=None,
    epochs=10,
    batch_size=50,
    lr=1e-3,
    use_cuda=True,
    contra_loss=False,
    margin=1.0,
    save_path="siamese.pkl",
):
    device = torch.device("cuda" if (use_cuda and torch.cuda.is_available()) else "cpu")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
    test_loader = None
    if test_dataset is not None:
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

    model = SiameseNetwork(contra_loss).to(device)

    if contra_loss:
        criterion = ContrastiveLoss(margin=margin)
    else:
        criterion = nn.BCELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(1, epochs + 1):
        running_loss = 0.0
        n_batches = 0

        for img1_set, img2_set, labels in train_loader:
            img1_set = img1_set.to(device)
            img2_set = img2_set.to(device)
            labels = torch.tensor(labels, dtype=torch.float32, device=device).view(-1, 1)

            optimizer.zero_grad()

            if contra_loss:
                out1, out2 = model(img1_set, img2_set)
                loss = criterion(out1, out2, labels)
            else:
                probs = model(img1_set, img2_set)      # (N,1) prob
                loss = criterion(probs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            n_batches += 1

        avg_loss = running_loss / max(n_batches, 1)
        print(f"Epoch {epoch}/{epochs} - loss: {avg_loss:.4f}")

    # Eval (si tienes tu función)
    if test_loader is not None:
        test_against_data(
            SimpleNamespace(cuda=(device.type == "cuda"), contra_loss=contra_loss, margin=margin),
            "testing",
            test_loader,
            model
        )

    torch.save(model.state_dict(), save_path)
    print("Model saved to:", save_path)
    return model


In [61]:
from types import SimpleNamespace

net = train_in_notebook(
    train_dataset=train_ds,
    test_dataset=test_ds,
    epochs=10,
    batch_size=50,
    lr=1e-3,
    use_cuda=True,
    contra_loss=False,   # pon True si quieres ContrastiveLoss
    margin=1.0,
    save_path="siamese.pkl",
)


C:\Users\Equipo\AppData\Local\Temp\ipykernel_18684\435712358.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels, dtype=torch.float32, device=device).view(-1, 1)


ValueError: expected 4D input (got 2D input)